# NB1 — Rayleigh/Rician Fading + Real ResNet-LSTM/Transformer-AMC Baselines + Entropy-Reg Attention Gate

Addresses:
- **Editor pt.1 / R4 pt.5 / R3 pt.3**: Rayleigh and Rician fading channels (with Doppler) added as held-out robustness test sets
- **R4 pt.2 / R3 pt.2**: ResNet-LSTM and Transformer-AMC trained from scratch on the same 36k AWGN dataset/split (lightweight, ~12hr budget); EMD-SVM also now measured, not estimated
- **R4 pt.1 / R3 pt.1**: Entropy-regularized cross-path attention gate, with automated adaptivity check (flags whether "SNR-adaptive" language is substantiated)

Run on Kaggle T4. Outputs saved to `/kaggle/working/` and exported via `nb1_export.pkl` + `model_D_seed*.weights.h5` for NB2/NB3.

In [ ]:
!pip install EMD-signal

In [ ]:
import os

# ── Output directory for all figures ─────────────────────────────────────
OUTPUT_DIR = '/kaggle/working/paper_figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Figures will be saved to: {os.path.abspath(OUTPUT_DIR)}')

# -*- coding: utf-8 -*-
"""
NB1 — Triple-Path AMC: Fading Channels + Real Baselines + Entropy-Reg Gate
Addresses (this notebook):
  - Editor pt.1, R4 pt.5, R3 pt.3: Rayleigh + Rician fading channels added
  - R4 pt.2, R3 pt.2: Real ResNet-LSTM and Transformer-AMC baselines trained
    (lightweight variants, same dataset/split, 12hr Kaggle budget)
  - R4 pt.1, R3 pt.1: Entropy-regularized cross-path attention gate
Strategy: AWGN-only training set unchanged in size (N_D=1500) but the
TEST SET now includes AWGN + Rayleigh + Rician conditions so robustness
can be reported per-channel. Baselines are lightweight (~150-300k params)
to keep total runtime under ~11.5 hrs on a single T4.
"""

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, mixed_precision
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.utils import to_categorical
from scipy.signal import hilbert, butter, lfilter
from scipy.stats import skew, kurtosis, entropy
from PyEMD import EMD
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import time
import gc
import logging
import warnings
logging.getLogger('PyEMD').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore')

# ── GPU setup ─────────────────────────────────────────────────────────────
mixed_precision.set_global_policy('mixed_float16')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU ready: {len(gpus)} device(s), mixed_float16 enabled")

# ══════════════════════════════════════════════════════════════════════════
# SCALE PARAMETERS
# ══════════════════════════════════════════════════════════════════════════
# Ablation variants B & C — moderate, fast, good enough to show progression
N_AB        = 1000
GASF_AB     = 64
SEQ_AB      = 128
BATCH_AB    = 128
EPOCHS_AB   = 60
PATIENCE_AB = 12

# Proposed Variant D — maximum accuracy
N_D         = 1500
GASF_D      = 96
SEQ_D       = 256
BATCH_D     = 64
EPOCHS_D    = 80
PATIENCE_D  = 20
N_ENSEMBLE  = 5

SNRs_to_test = [-10, -5, 0, 5, 10, 20]
CLASSES      = ['AM', 'PM', 'FM', 'SSB']
COLORS       = ['#2E4057', '#E84855', '#F4A261', '#48CAE4']

# ── Real baselines: scale params (lightweight, same dataset) ─────────────
# R4 pt.2, R3 pt.2: replace estimated ResNet-LSTM / Transformer-AMC with
# trained, lightweight reproductions on the SAME 36k AWGN dataset/split.
BASELINE_EPOCHS   = 40
BASELINE_PATIENCE = 8
BASELINE_BATCH    = 128

# ── Fading channel parameters (Editor pt.1, R4 pt.5, R3 pt.3) ─────────────
# Both Rayleigh and Rician fading are added to the TEST SET ONLY (the model
# is trained on AWGN as in the original submission; this evaluates
# generalization/robustness to unseen channel conditions, per R3's framing
# of "real-world evaluation").
RICIAN_K_FACTOR = 4.0   # dB, typical for pedestrian/vehicular line-of-sight
FADING_MAX_DOPPLER_HZ = 100.0  # ~ pedestrian/vehicular Doppler spread

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'figure.dpi': 120, 'savefig.dpi': 200,
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.15,
})

def butter_lowpass(cutoff, fs, order=9):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='low', analog=False)
    return b, a

# ══════════════════════════════════════════════════════════════════════════
# FADING CHANNEL MODELS (Rayleigh, Rician with Doppler)
# ══════════════════════════════════════════════════════════════════════════
def apply_rayleigh_fading(signal, fs, max_doppler_hz=FADING_MAX_DOPPLER_HZ, seed=None):
    """Apply frequency-selective Rayleigh fading via Jakes-like sum-of-sinusoids
    flat-fading multiplicative model (single-tap, time-varying envelope)."""
    rng = np.random.RandomState(seed)
    n = len(signal)
    t = np.arange(n) / fs
    n_osc = 16
    theta = rng.uniform(0, 2*np.pi, n_osc)
    phi_i = rng.uniform(0, 2*np.pi, n_osc)
    phi_q = rng.uniform(0, 2*np.pi, n_osc)
    h_i = np.zeros(n); h_q = np.zeros(n)
    for k in range(n_osc):
        fd = max_doppler_hz * np.cos(2*np.pi*k/n_osc + theta[0])
        h_i += np.cos(2*np.pi*fd*t + phi_i[k])
        h_q += np.cos(2*np.pi*fd*t + phi_q[k])
    h_i /= np.sqrt(n_osc); h_q /= np.sqrt(n_osc)
    envelope = np.sqrt(h_i**2 + h_q**2)
    envelope /= np.sqrt(np.mean(envelope**2))  # unit average power
    return signal * envelope

def apply_rician_fading(signal, fs, k_factor_db=RICIAN_K_FACTOR,
                         max_doppler_hz=FADING_MAX_DOPPLER_HZ, seed=None):
    """Rician fading: LOS component (K factor) + Rayleigh-distributed scatter."""
    rng = np.random.RandomState(seed)
    n = len(signal)
    t = np.arange(n) / fs
    K = 10**(k_factor_db/10)
    n_osc = 16
    theta = rng.uniform(0, 2*np.pi, n_osc)
    phi_i = rng.uniform(0, 2*np.pi, n_osc)
    phi_q = rng.uniform(0, 2*np.pi, n_osc)
    h_i = np.zeros(n); h_q = np.zeros(n)
    for k in range(n_osc):
        fd = max_doppler_hz * np.cos(2*np.pi*k/n_osc + theta[0])
        h_i += np.cos(2*np.pi*fd*t + phi_i[k])
        h_q += np.cos(2*np.pi*fd*t + phi_q[k])
    h_i /= np.sqrt(n_osc); h_q /= np.sqrt(n_osc)
    scatter_pow = 1.0 / (K + 1)
    los_pow     = K / (K + 1)
    h_i_n = h_i / np.sqrt(np.mean(h_i**2 + h_q**2)) * np.sqrt(2*scatter_pow)
    h_q_n = h_q / np.sqrt(np.mean(h_i**2 + h_q**2)) * np.sqrt(2*scatter_pow)
    los = np.sqrt(los_pow)
    envelope = np.sqrt((h_i_n + los)**2 + h_q_n**2)
    envelope /= np.sqrt(np.mean(envelope**2))
    return signal * envelope

print("Fading channel functions ready: Rayleigh, Rician "
      f"(K={RICIAN_K_FACTOR}dB, fD={FADING_MAX_DOPPLER_HZ}Hz)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 1. DATA GENERATION
# Train/test split uses AWGN as in original submission (N_D=1500/class/SNR).
# ADDITIONALLY generates Rayleigh- and Rician-faded versions of the TEST SET
# only, so all trained models can be evaluated for channel robustness
# (Editor pt.1, R4 pt.5, R3 pt.3) without altering the training distribution
# or requiring a full re-train.
# ══════════════════════════════════════════════════════════════════════════
def generate_signals(n_per_class, snrs=SNRs_to_test, num_samples=1024,
                      fading=None, seed_offset=0):
    """fading: None | 'rayleigh' | 'rician'. Fading applied to the modulated
    carrier BEFORE AWGN is added, then re-normalized (zero mean, unit var)."""
    total = n_per_class * 4 * len(snrs)
    print(f"Generating {total} signals ({n_per_class}/class/SNR, fading={fading})...")
    fs, fc = 100000, 5000
    t = np.arange(num_samples) / fs
    X, Y, SNR_labels = [], [], []
    sig_count = 0
    for snr in snrs:
        for _ in range(n_per_class):
            noise    = np.random.randn(num_samples)
            b, a     = butter_lowpass(1000, fs, order=9)
            m_t      = lfilter(b, a, noise)
            m_t_norm = m_t / np.max(np.abs(m_t))
            am   = (1 + 0.8*m_t_norm)*np.cos(2*np.pi*fc*t)
            pm   = np.cos(2*np.pi*fc*t + np.pi*m_t_norm)
            fm   = np.cos(2*np.pi*fc*t + 2*np.pi*1000*np.cumsum(m_t)/fs)
            m_hat = np.imag(hilbert(m_t))
            ssb  = m_t*np.cos(2*np.pi*fc*t) - m_hat*np.sin(2*np.pi*fc*t)
            for idx, sig in enumerate([am, pm, fm, ssb]):
                if fading == 'rayleigh':
                    sig = apply_rayleigh_fading(sig, fs, seed=seed_offset + sig_count)
                elif fading == 'rician':
                    sig = apply_rician_fading(sig, fs, seed=seed_offset + sig_count)
                sp  = np.mean(sig**2)
                np_ = sp / (10**(snr/10))
                ns  = sig + np.sqrt(np_)*np.random.randn(num_samples)
                X.append((ns - ns.mean()) / ns.std())
                Y.append(idx)
                SNR_labels.append(snr)
                sig_count += 1
    return (np.array(X, dtype=np.float32),
            np.array(Y, dtype=np.int8),
            np.array(SNR_labels, dtype=np.int8))

# Generate full D-scale AWGN dataset (training distribution unchanged)
X_raw, Y, SNR_labels = generate_signals(N_D, fading=None)

# ── Fading test sets: same per-class/SNR count as the test split will need.
# We generate a comparable-size set (20% of N_D, matching test_idx fraction)
# per fading type, purely for held-out robustness evaluation.
N_FADE_TEST = max(1, int(N_D * 0.2))
X_raw_rayleigh, Y_rayleigh, SNR_rayleigh = generate_signals(
    N_FADE_TEST, fading='rayleigh', seed_offset=10_000)
X_raw_rician, Y_rician, SNR_rician = generate_signals(
    N_FADE_TEST, fading='rician', seed_offset=20_000)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 2. FEATURE EXTRACTION — two resolutions simultaneously
#    Full-res (96x96, 256-step) for Variant D
#    Reduced-res (64x64, 128-step) for Variants B & C
#    Also extracted for Rayleigh/Rician test sets (full-res only, for D
#    and the new ResNet-LSTM/Transformer-AMC baselines)
# ══════════════════════════════════════════════════════════════════════════
def extract_emd_features(signal):
    emd  = EMD()
    imfs = emd.emd(signal, max_imf=3)
    features = []
    for i in range(3):
        if i < len(imfs):
            imf   = imfs[i]
            power = np.abs(imf)**2
            prob  = power / (np.sum(power) + 1e-10)
            features.extend([np.std(imf), skew(imf), kurtosis(imf),
                              np.sum(power), entropy(prob)])
        else:
            features.extend([0.0] * 5)
    return np.array(features, dtype=np.float32)

def compute_gasf(signal, size):
    sr     = np.interp(np.linspace(0, len(signal), size),
                       np.arange(len(signal)), signal)
    lo, hi = sr.min(), sr.max()
    n      = (sr - lo) / (hi - lo + 1e-8) * 2 - 1
    phi    = np.arccos(np.clip(n, -1, 1))
    c, s   = np.cos(phi), np.sin(phi)
    return (np.outer(c, c) - np.outer(s, s)).astype(np.float32)

def extract_sequence(signal, seq_len):
    stride = max(1, len(signal) // seq_len)
    seq    = signal[::stride][:seq_len]
    if len(seq) < seq_len:
        seq = np.pad(seq, (0, seq_len - len(seq)))
    return seq.astype(np.float32).reshape(seq_len, 1)

def extract_all_features(X_raw_arr, gasf_size, seq_len, tag=""):
    """Helper to extract EMD/GASF/Seq features for any signal array."""
    emd_l, gaf_l, seq_l = [], [], []
    for i, sig in enumerate(X_raw_arr):
        if i % 5000 == 0 and i > 0:
            print(f"  [{tag}] {i}/{len(X_raw_arr)} processed...")
        emd_l.append(extract_emd_features(sig))
        gaf_l.append(compute_gasf(sig, gasf_size))
        seq_l.append(extract_sequence(sig, seq_len))
    X_emd_ = np.array(emd_l, dtype=np.float32).reshape(-1, 15, 1)
    X_gaf_ = np.array(gaf_l, dtype=np.float32).reshape(-1, gasf_size, gasf_size, 1)
    X_seq_ = np.array(seq_l, dtype=np.float32)
    return X_emd_, X_gaf_, X_seq_

print(f"\nExtracting features for {len(X_raw)} AWGN signals...")
print(f"  EMD: 15 features  |  GASF_D: {GASF_D}x{GASF_D}  |  "
      f"GASF_AB: {GASF_AB}x{GASF_AB}  |  SeqD: {SEQ_D}  |  SeqAB: {SEQ_AB}")

emd_data  = []
gaf_d     = []   # full-res for D
gaf_ab    = []   # reduced-res for B & C
seq_d     = []   # full-length for D
seq_ab    = []   # shorter for B & C

for i, sig in enumerate(X_raw):
    if i % 5000 == 0 and i > 0:
        print(f"  {i}/{len(X_raw)} processed...")
    emd_data.append(extract_emd_features(sig))
    gaf_d.append(compute_gasf(sig, GASF_D))
    gaf_ab.append(compute_gasf(sig, GASF_AB))
    seq_d.append(extract_sequence(sig, SEQ_D))
    seq_ab.append(extract_sequence(sig, SEQ_AB))

print("Clearing raw AWGN signals...")
del X_raw
gc.collect()

X_emd   = np.array(emd_data,  dtype=np.float32).reshape(-1, 15, 1)
X_gaf_d = np.array(gaf_d,     dtype=np.float32).reshape(-1, GASF_D,  GASF_D,  1)
X_gaf_ab= np.array(gaf_ab,    dtype=np.float32).reshape(-1, GASF_AB, GASF_AB, 1)
X_seq_d = np.array(seq_d,     dtype=np.float32)
X_seq_ab= np.array(seq_ab,    dtype=np.float32)

del emd_data, gaf_d, gaf_ab, seq_d, seq_ab
gc.collect()

# ── Train/test split on FULL AWGN dataset ─────────────────────────────────
Y_onehot = to_categorical(Y, num_classes=4)
train_idx, test_idx = train_test_split(
    np.arange(len(Y)), test_size=0.2, random_state=42, stratify=Y)

# ── For B & C: use only N_AB/N_D fraction of training data ───────────────
ab_frac   = N_AB / N_D
n_ab_train = int(len(train_idx) * ab_frac)
np.random.seed(42)
ab_train_idx = np.random.choice(train_idx, n_ab_train, replace=False)

# B & C data (smaller GASF, subsampled training)
train_ab = {"emd_input": X_emd[ab_train_idx],
            "gaf_input": X_gaf_ab[ab_train_idx]}
test_ab  = {"emd_input": X_emd[test_idx],
            "gaf_input": X_gaf_ab[test_idx]}
y_train_ab = Y_onehot[ab_train_idx]

# D data (full GASF, full training set)
train_d  = {"emd_input": X_emd[train_idx],
            "gaf_input": X_gaf_d[train_idx],
            "seq_input": X_seq_d[train_idx]}
test_d   = {"emd_input": X_emd[test_idx],
            "gaf_input": X_gaf_d[test_idx],
            "seq_input": X_seq_d[test_idx]}
y_train_d  = Y_onehot[train_idx]
y_test     = Y_onehot[test_idx]
snr_test   = SNR_labels[test_idx]

# EMD flat for sklearn baselines
X_emd_flat = X_emd.reshape(-1, 15)
X_tr_emd   = X_emd_flat[train_idx]
X_te_emd   = X_emd_flat[test_idx]

del X_emd, X_gaf_d, X_gaf_ab, X_seq_d, X_seq_ab, Y, Y_onehot
gc.collect()
print(f"Data ready. D-train: {len(train_idx)} | AB-train: {n_ab_train} | Test (AWGN): {len(test_idx)}")

# ══════════════════════════════════════════════════════════════════════════
# FADING TEST SETS — features (full-res D config only)
# ══════════════════════════════════════════════════════════════════════════
print(f"\nExtracting features for Rayleigh test set ({len(X_raw_rayleigh)} signals)...")
X_emd_ray, X_gaf_ray, X_seq_ray = extract_all_features(
    X_raw_rayleigh, GASF_D, SEQ_D, tag="Rayleigh")
y_rayleigh = to_categorical(Y_rayleigh, num_classes=4)
test_rayleigh = {"emd_input": X_emd_ray, "gaf_input": X_gaf_ray, "seq_input": X_seq_ray}
del X_raw_rayleigh; gc.collect()

print(f"\nExtracting features for Rician test set ({len(X_raw_rician)} signals)...")
X_emd_ric, X_gaf_ric, X_seq_ric = extract_all_features(
    X_raw_rician, GASF_D, SEQ_D, tag="Rician")
y_rician = to_categorical(Y_rician, num_classes=4)
test_rician = {"emd_input": X_emd_ric, "gaf_input": X_gaf_ric, "seq_input": X_seq_ric}
del X_raw_rician; gc.collect()

print(f"Fading test sets ready. Rayleigh: {len(Y_rayleigh)} | Rician: {len(Y_rician)}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 3. SHARED BUILDING BLOCKS + MODEL ARCHITECTURES
# ══════════════════════════════════════════════════════════════════════════
def se_block(x, ratio=8):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Reshape((1, 1, filters))(se)
    se = layers.Dense(max(filters//ratio, 1), activation='relu',
                      kernel_initializer='he_normal', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid',
                      kernel_initializer='he_normal', use_bias=False)(se)
    return layers.Multiply()([x, se])

class CosineAnnealingLR(tf.keras.callbacks.Callback):
    def __init__(self, initial_lr=0.0005, min_lr=1e-6, total_epochs=80):
        super().__init__()
        self.initial_lr = initial_lr
        self.min_lr     = min_lr
        self.total_epochs = total_epochs
    def on_epoch_begin(self, epoch, logs=None):
        decay  = 0.5*(1 + np.cos(np.pi*epoch/self.total_epochs))
        new_lr = self.min_lr + (self.initial_lr - self.min_lr)*decay
        self.model.optimizer.learning_rate.assign(float(new_lr))

def get_callbacks_ab():
    return [
        CosineAnnealingLR(0.0005, 1e-6, EPOCHS_AB),
        ReduceLROnPlateau('val_loss', factor=0.5, patience=6,
                          min_lr=1e-6, verbose=0),
        EarlyStopping('val_loss', patience=PATIENCE_AB,
                      restore_best_weights=True)
    ]

def get_callbacks_d():
    return [
        CosineAnnealingLR(0.0005, 1e-6, EPOCHS_D),
        ReduceLROnPlateau('val_loss', factor=0.5, patience=8,
                          min_lr=1e-6, verbose=0),
        EarlyStopping('val_loss', patience=PATIENCE_D,
                      restore_best_weights=True)
    ]

def get_callbacks_baseline():
    return [
        CosineAnnealingLR(0.0005, 1e-6, BASELINE_EPOCHS),
        ReduceLROnPlateau('val_loss', factor=0.5, patience=5,
                          min_lr=1e-6, verbose=0),
        EarlyStopping('val_loss', patience=BASELINE_PATIENCE,
                      restore_best_weights=True)
    ]

def gaf_cnn_branch(gaf_input, gasf_size, dual_kernel=True):
    if dual_kernel:
        x = layers.Concatenate()([
            layers.Conv2D(16,(3,3),padding='same',activation='relu')(gaf_input),
            layers.Conv2D(16,(5,5),padding='same',activation='relu')(gaf_input),
        ])
    else:
        x = layers.Conv2D(32,(3,3),padding='same',activation='relu')(gaf_input)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = se_block(x)
    x = layers.Conv2D(64,(3,3),padding='same',activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = se_block(x)
    n_filters = 128 if gasf_size == GASF_D else 64
    x = layers.Conv2D(n_filters,(3,3),padding='same',activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = se_block(x)
    return layers.Concatenate()([layers.GlobalAveragePooling2D()(x),
                                  layers.GlobalMaxPooling2D()(x)])

# ══════════════════════════════════════════════════════════════════════════
# MODEL B: Single-kernel GASF + EMD  (ablation — moderate scale)
# ══════════════════════════════════════════════════════════════════════════
def build_single_kernel():
    emd_input = layers.Input(shape=(15,1), name="emd_input")
    gaf_input = layers.Input(shape=(GASF_AB, GASF_AB, 1), name="gaf_input")
    emd_raw   = layers.Flatten()(emd_input)
    x1 = layers.Dense(64, activation='relu')(emd_raw)
    x1 = layers.Dense(32, activation='relu')(x1)
    x2 = gaf_cnn_branch(gaf_input, GASF_AB, dual_kernel=False)
    merged = layers.Concatenate()([x1, x2, emd_raw])
    z = layers.Dense(128, activation='relu')(merged)
    z = layers.Dropout(0.2)(z)
    z = layers.Dense(64, activation='relu')(z)
    out = layers.Dense(4, activation='softmax', dtype='float32')(z)
    model = Model(inputs=[emd_input, gaf_input], outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(0.0005),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ══════════════════════════════════════════════════════════════════════════
# MODEL C: Dual-kernel GASF + EMD  (ablation — moderate scale)
# ══════════════════════════════════════════════════════════════════════════
def build_dual_kernel():
    emd_input = layers.Input(shape=(15,1), name="emd_input")
    gaf_input = layers.Input(shape=(GASF_AB, GASF_AB, 1), name="gaf_input")
    emd_raw   = layers.Flatten()(emd_input)
    x1 = layers.Dense(64, activation='relu')(emd_raw)
    x1 = layers.Dense(32, activation='relu')(x1)
    x2 = gaf_cnn_branch(gaf_input, GASF_AB, dual_kernel=True)
    merged = layers.Concatenate()([x1, x2, emd_raw])
    z = layers.Dense(128, activation='relu')(merged)
    z = layers.Dropout(0.2)(z)
    z = layers.Dense(64, activation='relu')(z)
    out = layers.Dense(4, activation='softmax', dtype='float32')(z)
    model = Model(inputs=[emd_input, gaf_input], outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(0.0005),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ══════════════════════════════════════════════════════════════════════════
# ENTROPY-REGULARIZED CROSS-PATH ATTENTION GATE  (R4 pt.1, R3 pt.1)
# ──────────────────────────────────────────────────────────────────────────
# The original gate collapsed to fixed weights (a=[0, 0.6, 0.4]) at every
# SNR because it was trained jointly with a single cross-entropy loss and
# had no incentive to vary with input statistics. Per R3 pt.1, we add an
# entropy regularization term on the gate's softmax output: instead of
# penalizing low entropy (which would push toward MORE uniform/deterministic
# behaviour), here we ADD a custom loss term that penalizes the gate's
# weights for being CONSTANT across a batch (variance-encouraging term),
# directly encouraging SNR-conditional / input-conditional gating.
#
# Implementation: a custom Keras loss wrapper combines categorical
# cross-entropy with -lambda * Var(gate_weights across batch). Maximizing
# variance of the gate output across the batch (which contains a mix of
# SNRs) directly counteracts gate collapse to a single fixed point.
# ══════════════════════════════════════════════════════════════════════════
GATE_ENTROPY_LAMBDA = 0.05  # weight on the variance-encouraging term

class EntropyRegGateLayer(layers.Layer):
    """Wraps the path_gate Dense+softmax and stores its output for the
    custom loss to consume via add_loss."""
    def __init__(self, lam=GATE_ENTROPY_LAMBDA, **kwargs):
        super().__init__(**kwargs)
        self.lam = lam
        self.dense = layers.Dense(3, activation='softmax', name='path_gate_raw',
                                   dtype='float32')

    def call(self, inputs):
        gate = self.dense(inputs)
        # Variance of each gate channel across the batch — we want this to be
        # LARGE (gate should respond differently to different inputs), so we
        # ADD a penalty equal to -lambda * mean(variance) to total loss.
        # tf.reduce_variance not available -> compute manually
        mean_g = tf.reduce_mean(gate, axis=0, keepdims=True)
        var_g  = tf.reduce_mean(tf.square(gate - mean_g), axis=0)  # (3,)
        collapse_penalty = self.lam * tf.exp(-10.0 * tf.reduce_mean(var_g))
        self.add_loss(collapse_penalty)
        return gate

# ══════════════════════════════════════════════════════════════════════════
# MODEL D: Triple-Path  (PROPOSED — with entropy-regularized gate)
# ══════════════════════════════════════════════════════════════════════════
def build_triple_path(entropy_reg=True):
    # ── Path 1: EMD Statistical ──────────────────────────────────────────
    emd_input = layers.Input(shape=(15,1),  name="emd_input")
    emd_raw   = layers.Flatten()(emd_input)
    x1 = layers.Dense(64, activation='relu')(emd_raw)
    x1 = layers.BatchNormalization()(x1)
    x1 = layers.Dense(32, activation='relu')(x1)

    # ── Path 2: Dual-kernel GASF-CNN (full 96x96, 128 filters) ──────────
    gaf_input = layers.Input(shape=(GASF_D, GASF_D, 1), name="gaf_input")
    x2 = gaf_cnn_branch(gaf_input, GASF_D, dual_kernel=True)

    # ── Path 3: Conv1D + Stacked BiGRU (full SEQ_D=256) ─────────────────
    seq_input = layers.Input(shape=(SEQ_D, 1), name="seq_input")
    x3 = layers.Conv1D(32, 7, strides=2, padding='same',
                        activation='relu')(seq_input)
    x3 = layers.BatchNormalization()(x3)
    x3 = layers.Conv1D(64, 5, strides=2, padding='same',
                        activation='relu')(x3)
    x3 = layers.BatchNormalization()(x3)
    x3 = layers.Conv1D(64, 3, strides=1, padding='same',
                        activation='relu')(x3)
    x3 = layers.BatchNormalization()(x3)
    x3 = layers.Bidirectional(
        layers.GRU(64, return_sequences=True,  dropout=0.1,
                   recurrent_dropout=0.0))(x3)
    x3 = layers.Bidirectional(
        layers.GRU(32, return_sequences=False, dropout=0.1,
                   recurrent_dropout=0.0))(x3)

    # ── Cross-Path Attention Gate (entropy-regularized) ──────────────────
    concat_all = layers.Concatenate()([x1, x2, x3, emd_raw])
    gate_hidden = layers.Dense(128, activation='relu')(concat_all)
    if entropy_reg:
        gate = EntropyRegGateLayer(name='path_gate')(gate_hidden)
    else:
        gate = layers.Dense(3, activation='softmax', name='path_gate')(gate_hidden)

    x1_g = layers.Lambda(
        lambda inp: inp[0] * tf.expand_dims(inp[1][:,0], 1),
        name='gate_emd')([x1, gate])
    x2_g = layers.Lambda(
        lambda inp: inp[0] * tf.expand_dims(inp[1][:,1], 1),
        name='gate_gaf')([x2, gate])
    x3_g = layers.Lambda(
        lambda inp: inp[0] * tf.expand_dims(inp[1][:,2], 1),
        name='gate_gru')([x3, gate])

    # ── Fusion Head ───────────────────────────────────────────────────────
    merged = layers.Concatenate()([x1_g, x2_g, x3_g, emd_raw])
    z = layers.Dense(256, activation='relu')(merged)
    z = layers.BatchNormalization()(z)
    z = layers.Dropout(0.25)(z)
    z = layers.Dense(128, activation='relu')(z)
    z = layers.Dropout(0.15)(z)
    z = layers.Dense(64,  activation='relu')(z)
    out = layers.Dense(4, activation='softmax', dtype='float32')(z)

    model = Model(inputs=[emd_input, gaf_input, seq_input], outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=['accuracy']
    )
    return model

# ══════════════════════════════════════════════════════════════════════════
# REAL BASELINE 1: ResNet-LSTM  (R4 pt.2, R3 pt.2)
# ──────────────────────────────────────────────────────────────────────────
# Lightweight residual CNN (on GASF_AB images) feeding an LSTM over the
# sequence input. Operates on the SAME emd/gaf_ab/seq_ab-style inputs as
# Variants B/C for parity, but uses raw GASF + 1D sequence (no EMD stats),
# matching the "ResNet-LSTM" family used in AMC literature: a residual CNN
# feature extractor + recurrent temporal head.
# ══════════════════════════════════════════════════════════════════════════
def resnet_block_1d(x, filters, kernel_size=5):
    shortcut = x
    y = layers.Conv1D(filters, kernel_size, padding='same')(x)
    y = layers.BatchNormalization()(y)
    y = layers.Activation('relu')(y)
    y = layers.Conv1D(filters, kernel_size, padding='same')(y)
    y = layers.BatchNormalization()(y)
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, padding='same')(shortcut)
    y = layers.Add()([y, shortcut])
    y = layers.Activation('relu')(y)
    return y

def build_resnet_lstm():
    seq_input = layers.Input(shape=(SEQ_D, 1), name="seq_input")
    x = layers.Conv1D(32, 7, strides=2, padding='same', activation='relu')(seq_input)
    x = layers.BatchNormalization()(x)
    x = resnet_block_1d(x, 32)
    x = layers.MaxPooling1D(2)(x)
    x = resnet_block_1d(x, 64)
    x = layers.MaxPooling1D(2)(x)
    x = layers.LSTM(64, return_sequences=False, dropout=0.1)(x)
    z = layers.Dense(64, activation='relu')(x)
    z = layers.Dropout(0.2)(z)
    out = layers.Dense(4, activation='softmax', dtype='float32')(z)
    model = Model(inputs=seq_input, outputs=out, name="ResNet_LSTM")
    model.compile(optimizer=tf.keras.optimizers.Adam(0.0005),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ══════════════════════════════════════════════════════════════════════════
# REAL BASELINE 2: Transformer-AMC  (R4 pt.2, R3 pt.2)
# ──────────────────────────────────────────────────────────────────────────
# Lightweight Transformer encoder operating on the 1D sequence input,
# matching "Transformer-AMC"-style architectures in the literature: patch
# embedding + multi-head self-attention encoder blocks + classification head.
# ══════════════════════════════════════════════════════════════════════════
def transformer_encoder_block(x, head_size, num_heads, ff_dim, dropout=0.1):
    attn = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads,
                                       dropout=dropout)(x, x)
    attn = layers.Dropout(dropout)(attn)
    x = layers.Add()([x, attn])
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    ff = layers.Dense(ff_dim, activation='relu')(x)
    ff = layers.Dense(x.shape[-1])(ff)
    ff = layers.Dropout(dropout)(ff)
    x = layers.Add()([x, ff])
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return x

def build_transformer_amc(patch_len=16, d_model=64, num_heads=4,
                           ff_dim=128, num_blocks=2):
    seq_input = layers.Input(shape=(SEQ_D, 1), name="seq_input")
    n_patches = SEQ_D // patch_len
    x = layers.Reshape((n_patches, patch_len))(seq_input)
    x = layers.Dense(d_model)(x)
    # learnable positional embedding
    pos_emb = layers.Embedding(input_dim=n_patches, output_dim=d_model)(
        tf.range(start=0, limit=n_patches, delta=1))
    x = x + pos_emb
    for _ in range(num_blocks):
        x = transformer_encoder_block(x, head_size=d_model//num_heads,
                                        num_heads=num_heads, ff_dim=ff_dim)
    x = layers.GlobalAveragePooling1D()(x)
    z = layers.Dense(64, activation='relu')(x)
    z = layers.Dropout(0.2)(z)
    out = layers.Dense(4, activation='softmax', dtype='float32')(z)
    model = Model(inputs=seq_input, outputs=out, name="Transformer_AMC")
    model.compile(optimizer=tf.keras.optimizers.Adam(0.0005),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

print("Models defined: Single-Kernel, Dual-Kernel, Triple-Path (entropy-reg gate), "
      "ResNet-LSTM, Transformer-AMC")
print(f"Entropy regularization lambda = {GATE_ENTROPY_LAMBDA}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 4. TRAINING
# ══════════════════════════════════════════════════════════════════════════
def zone_predict(models, data, snr):
    temperature = 0.7 if snr <= -5 else (0.5 if snr == 0 else 0.3)
    all_logits  = []
    for m in models:
        preds  = m.predict(data, verbose=0)
        logits = np.log(preds + 1e-10) / temperature
        all_logits.append(logits)
    avg = np.mean(all_logits, axis=0)
    exp = np.exp(avg - np.max(avg, axis=1, keepdims=True))
    return exp / np.sum(exp, axis=1, keepdims=True)

def eval_per_snr(models, td, yt, snr_labels, triple=False):
    accs = []
    for snr in SNRs_to_test:
        idx  = np.where(snr_labels == snr)[0]
        data = {"emd_input": td["emd_input"][idx],
                "gaf_input": td["gaf_input"][idx]}
        if triple:
            data["seq_input"] = td["seq_input"][idx]
        pr  = zone_predict(models, data, snr)
        acc = np.mean(np.argmax(pr,1) == np.argmax(yt[idx],1)) * 100
        accs.append(acc)
    return accs

def print_header(name, desc, n):
    print("\n" + "█"*65)
    print(f"█  {name}")
    print(f"█  {desc}")
    print(f"█  {n} models")
    print("█"*65)

def print_done(i, n, h):
    ep  = len(h.history['loss'])
    acc = h.history['val_accuracy'][-1]*100
    best_acc = max(h.history['val_accuracy'])*100
    print(f"  └─ Model {i}/{n} | Epochs: {ep} | "
          f"Final val acc: {acc:.1f}% | Best val acc: {best_acc:.1f}%")

def print_results(name, accs):
    print(f"\n  ╔══ {name} ══╗")
    for snr, acc in zip(SNRs_to_test, accs):
        bar = '█' * int(acc/5)
        print(f"  ║  {snr:>4} dB : {acc:>6.2f}%  {bar}")
    print(f"  ║  Mean: {np.mean(accs):.2f}%")
    print(f"  ╚{'═'*42}╝")

results = {}

# ── Variant B ─────────────────────────────────────────────────────────────
print_header("VARIANT B — Single-Kernel GASF+EMD",
             f"Ablation | {GASF_AB}x{GASF_AB} GASF | {n_ab_train} train samples", 2)
models_B, t0 = [], time.time()
for i in range(2):
    print(f"  ┌─ Model {i+1}/2...")
    m = build_single_kernel()
    h = m.fit(train_ab, y_train_ab,
              validation_data=(test_ab, y_test),
              batch_size=BATCH_AB, epochs=EPOCHS_AB,
              callbacks=get_callbacks_ab(), verbose=1)
    print_done(i+1, 2, h); models_B.append(m)
results['Single-Kernel GASF+EMD'] = eval_per_snr(
    models_B, test_ab, y_test, snr_test)
print_results("Single-Kernel GASF+EMD", results['Single-Kernel GASF+EMD'])
print(f"  ⏱  {(time.time()-t0)/60:.1f} min")
tf.keras.backend.clear_session(); gc.collect()

# ── Variant C ─────────────────────────────────────────────────────────────
print_header("VARIANT C — Dual-Kernel GASF+EMD",
             f"Ablation | {GASF_AB}x{GASF_AB} GASF | {n_ab_train} train samples", 2)
models_C, t0 = [], time.time()
for i in range(2):
    print(f"  ┌─ Model {i+1}/2...")
    m = build_dual_kernel()
    h = m.fit(train_ab, y_train_ab,
              validation_data=(test_ab, y_test),
              batch_size=BATCH_AB, epochs=EPOCHS_AB,
              callbacks=get_callbacks_ab(), verbose=1)
    print_done(i+1, 2, h); models_C.append(m)
results['Dual-Kernel GASF+EMD'] = eval_per_snr(
    models_C, test_ab, y_test, snr_test)
print_results("Dual-Kernel GASF+EMD", results['Dual-Kernel GASF+EMD'])
print(f"  ⏱  {(time.time()-t0)/60:.1f} min")
tf.keras.backend.clear_session(); gc.collect()

# ── Variant D: PROPOSED (entropy-regularized gate) ────────────────────────
print_header("VARIANT D — ★ PROPOSED: EMD + GASF + BiGRU (entropy-reg gate) ★",
             f"FULL SCALE | {GASF_D}x{GASF_D} GASF | {SEQ_D}-step BiGRU | "
             f"{len(train_idx)} train samples | label smoothing 0.05 | "
             f"gate entropy λ={GATE_ENTROPY_LAMBDA}",
             N_ENSEMBLE)
models_D, t0 = [], time.time()
for i in range(N_ENSEMBLE):
    print(f"  ┌─ Model {i+1}/{N_ENSEMBLE}...")
    tf.random.set_seed(i * 42)
    np.random.seed(i * 42)
    m = build_triple_path(entropy_reg=True)
    h = m.fit(train_d, y_train_d,
              validation_data=(test_d, y_test),
              batch_size=BATCH_D, epochs=EPOCHS_D,
              callbacks=get_callbacks_d(), verbose=1)
    print_done(i+1, N_ENSEMBLE, h)
    models_D.append(m)
results['Proposed: EMD+GASF+BiGRU'] = eval_per_snr(
    models_D, test_d, y_test, snr_test, triple=True)
print_results("★ Proposed: EMD+GASF+BiGRU ★", results['Proposed: EMD+GASF+BiGRU'])
print(f"  ⏱  {(time.time()-t0)/60:.1f} min")

# ══════════════════════════════════════════════════════════════════════════
# REAL BASELINES: ResNet-LSTM and Transformer-AMC  (R4 pt.2, R3 pt.2)
# Trained on the SAME train_d/y_train_d (seq_input only), evaluated on the
# SAME test split, SNR levels, and 4 modulation classes.
# ══════════════════════════════════════════════════════════════════════════
print_header("REAL BASELINE — ResNet-LSTM",
             f"{BASELINE_EPOCHS} max epochs | seq_input ({SEQ_D},1) | same split", 1)
t0 = time.time()
m_resnet_lstm = build_resnet_lstm()
h = m_resnet_lstm.fit(
    {"seq_input": train_d["seq_input"]}, y_train_d,
    validation_data=({"seq_input": test_d["seq_input"]}, y_test),
    batch_size=BASELINE_BATCH, epochs=BASELINE_EPOCHS,
    callbacks=get_callbacks_baseline(), verbose=1)
print_done(1, 1, h)
accs_resnet_lstm = []
for snr in SNRs_to_test:
    idx = np.where(snr_test == snr)[0]
    pr  = m_resnet_lstm.predict({"seq_input": test_d["seq_input"][idx]}, verbose=0)
    accs_resnet_lstm.append(np.mean(np.argmax(pr,1)==np.argmax(y_test[idx],1))*100)
results['ResNet-LSTM (reproduced)'] = accs_resnet_lstm
print_results("ResNet-LSTM (reproduced)", accs_resnet_lstm)
print(f"  ⏱  {(time.time()-t0)/60:.1f} min | Params: {m_resnet_lstm.count_params():,}")

# Fading eval for ResNet-LSTM (before session clear)
pr_ray = m_resnet_lstm.predict({"seq_input": test_rayleigh["seq_input"]}, verbose=0)
pr_ric = m_resnet_lstm.predict({"seq_input": test_rician["seq_input"]}, verbose=0)
acc_resnet_ray = np.mean(np.argmax(pr_ray,1)==np.argmax(y_rayleigh,1))*100
acc_resnet_ric = np.mean(np.argmax(pr_ric,1)==np.argmax(y_rician,1))*100
m_resnet_lstm_params = m_resnet_lstm.count_params()
tf.keras.backend.clear_session(); gc.collect()

print_header("REAL BASELINE — Transformer-AMC",
             f"{BASELINE_EPOCHS} max epochs | seq_input ({SEQ_D},1) | same split", 1)
t0 = time.time()
m_transformer = build_transformer_amc()
h = m_transformer.fit(
    {"seq_input": train_d["seq_input"]}, y_train_d,
    validation_data=({"seq_input": test_d["seq_input"]}, y_test),
    batch_size=BASELINE_BATCH, epochs=BASELINE_EPOCHS,
    callbacks=get_callbacks_baseline(), verbose=1)
print_done(1, 1, h)
accs_transformer = []
for snr in SNRs_to_test:
    idx = np.where(snr_test == snr)[0]
    pr  = m_transformer.predict({"seq_input": test_d["seq_input"][idx]}, verbose=0)
    accs_transformer.append(np.mean(np.argmax(pr,1)==np.argmax(y_test[idx],1))*100)
results['Transformer-AMC (reproduced)'] = accs_transformer
print_results("Transformer-AMC (reproduced)", accs_transformer)
print(f"  ⏱  {(time.time()-t0)/60:.1f} min | Params: {m_transformer.count_params():,}")

# Fading eval for Transformer-AMC (before session clear)
pr_ray = m_transformer.predict({"seq_input": test_rayleigh["seq_input"]}, verbose=0)
pr_ric = m_transformer.predict({"seq_input": test_rician["seq_input"]}, verbose=0)
acc_transformer_ray = np.mean(np.argmax(pr_ray,1)==np.argmax(y_rayleigh,1))*100
acc_transformer_ric = np.mean(np.argmax(pr_ric,1)==np.argmax(y_rician,1))*100
m_transformer_params = m_transformer.count_params()
tf.keras.backend.clear_session(); gc.collect()

# ── Sklearn Baselines ─────────────────────────────────────────────────────
print_header("SKLEARN BASELINES", "kNN (k=5) + MLP-ANN on EMD features", 2)
scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_tr_emd)
X_te_sc  = scaler.transform(X_te_emd)
y_tr_int = np.argmax(y_train_d, axis=1)

for clf_name, clf in [
    ('kNN (k=5)',  KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
    ('MLP-ANN',    MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300,
                                  random_state=42, early_stopping=True,
                                  validation_fraction=0.1)),
]:
    print(f"\n  ┌─ Training {clf_name}...")
    t1 = time.time()
    clf.fit(X_tr_sc, y_tr_int)
    accs = []
    for snr in SNRs_to_test:
        idx  = np.where(snr_test == snr)[0]
        pred = clf.predict(X_te_sc[idx])
        true = np.argmax(y_test[idx], axis=1)
        accs.append(accuracy_score(true, pred) * 100)
    results[clf_name] = accs
    print(f"  └─ Done in {time.time()-t1:.1f}s")
    print_results(clf_name, accs)

# ══════════════════════════════════════════════════════════════════════════
# EMD-SVM BASELINE — trained on the SAME data (R3 pt.2: no more "estimated")
# ══════════════════════════════════════════════════════════════════════════
from sklearn.svm import SVC
print_header("EMD-SVM BASELINE", "SVM (RBF) on EMD features — measured, not estimated", 1)
t1 = time.time()
svm_clf = SVC(kernel='rbf', C=10, gamma='scale')
svm_clf.fit(X_tr_sc, y_tr_int)
accs_svm = []
for snr in SNRs_to_test:
    idx  = np.where(snr_test == snr)[0]
    pred = svm_clf.predict(X_te_sc[idx])
    true = np.argmax(y_test[idx], axis=1)
    accs_svm.append(accuracy_score(true, pred) * 100)
results['EMD-SVM (Baseline [1], measured)'] = accs_svm
print_results("EMD-SVM (measured)", accs_svm)
print(f"  ⏱  {(time.time()-t1)/60:.1f} min")

# ══════════════════════════════════════════════════════════════════════════
# FADING CHANNEL ROBUSTNESS EVALUATION  (Editor pt.1, R4 pt.5, R3 pt.3)
# Proposed (ensemble), ResNet-LSTM, Transformer-AMC evaluated on Rayleigh
# and Rician test sets (model trained on AWGN only — generalization test)
# ══════════════════════════════════════════════════════════════════════════
print_header("FADING CHANNEL ROBUSTNESS",
             "Proposed / ResNet-LSTM / Transformer-AMC on Rayleigh & Rician", 1)

def eval_fading_proposed(models, fade_data, fade_y):
    """Proposed ensemble doesn't have per-sample SNR for fading set; use
    a fixed mid-range temperature (T=0.5) since SNR-zone is unknown."""
    all_logits = []
    for m in models:
        pr = m.predict(fade_data, verbose=0)
        all_logits.append(np.log(pr + 1e-10) / 0.5)
    avg = np.mean(all_logits, axis=0)
    exp = np.exp(avg - np.max(avg, axis=1, keepdims=True))
    pr  = exp / np.sum(exp, axis=1, keepdims=True)
    return np.mean(np.argmax(pr,1) == np.argmax(fade_y,1)) * 100

fading_results = {}
fading_results['Proposed (AWGN-trained) — Rayleigh'] = eval_fading_proposed(
    models_D, test_rayleigh, y_rayleigh)
fading_results['Proposed (AWGN-trained) — Rician'] = eval_fading_proposed(
    models_D, test_rician, y_rician)
fading_results['ResNet-LSTM (AWGN-trained) — Rayleigh'] = acc_resnet_ray
fading_results['ResNet-LSTM (AWGN-trained) — Rician']   = acc_resnet_ric
fading_results['Transformer-AMC (AWGN-trained) — Rayleigh'] = acc_transformer_ray
fading_results['Transformer-AMC (AWGN-trained) — Rician']   = acc_transformer_ric

print("\n" + "═"*65)
print("FADING CHANNEL RESULTS (AWGN-trained models, held-out fading test)")
print("Editor pt.1, R4 pt.5, R3 pt.3 — multipath fading robustness")
print("═"*65)
for k, v in fading_results.items():
    print(f"  {k:<45} {v:>6.2f}%")
print(f"\n  Reference — Proposed AWGN mean accuracy (all SNRs): "
      f"{np.mean(results['Proposed: EMD+GASF+BiGRU']):.2f}%")
print("═"*65)

# ── Final Summary ─────────────────────────────────────────────────────────
order = ['EMD-SVM (Baseline [1], measured)', 'kNN (k=5)', 'MLP-ANN',
         'ResNet-LSTM (reproduced)', 'Transformer-AMC (reproduced)',
         'Single-Kernel GASF+EMD', 'Dual-Kernel GASF+EMD',
         'Proposed: EMD+GASF+BiGRU']
print("\n" + "═"*78)
print(f"{'METHOD':<35}" + "".join([f"{s:>7} dB" for s in SNRs_to_test]))
print("─"*78)
for name in order:
    if name not in results: continue
    tag = '  ◄ PROPOSED' if 'Proposed' in name else \
          '  ◄ BASELINE' if 'Baseline' in name or 'reproduced' in name else ''
    print(f"{name:<35}" +
          "".join([f"{a:>9.2f}" for a in results[name]]) + tag)
print("═"*78)
print(f"\nParameter counts:")
print(f"  Single-Kernel GASF+EMD: {models_B[0].count_params():>10,}")
print(f"  Dual-Kernel GASF+EMD:   {models_C[0].count_params():>10,}")
print(f"  Proposed (Triple-Path): {models_D[0].count_params():>10,}")
print(f"  ResNet-LSTM:            {m_resnet_lstm_params:>10,}")
print(f"  Transformer-AMC:        {m_transformer_params:>10,}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CROSS-PATH ATTENTION WEIGHT VISUALIZATION — entropy-regularized gate
# (R4 pt.1, R3 pt.1)
# Reports whether the entropy-regularization term produced SNR-dependent
# gate weights. If the weights remain ~constant across SNR (variance below
# a small threshold), the manuscript text is flagged for revision to
# "learned fixed path weighting" rather than "SNR-adaptive".
# ══════════════════════════════════════════════════════════════════════════
import tensorflow as tf

gate_layer_name = 'path_gate'
try:
    test_layer = models_D[0].get_layer(gate_layer_name)
    gate_model_ok = True
except Exception:
    gate_model_ok = False

if gate_model_ok:
    mean_weights = {}
    for snr in SNRs_to_test:
        idx  = np.where(snr_test == snr)[0]
        data = {"emd_input": test_d["emd_input"][idx],
                "gaf_input": test_d["gaf_input"][idx],
                "seq_input": test_d["seq_input"][idx]}
        all_gates = []
        for m in models_D:
            gm = tf.keras.Model(inputs=m.inputs, outputs=m.get_layer(gate_layer_name).output)
            gw = gm.predict(data, verbose=0)
            all_gates.append(gw)
        mean_gate = np.mean(all_gates, axis=0)
        mean_weights[snr] = mean_gate.mean(axis=0)

    snr_labels_w = [str(s) for s in SNRs_to_test]
    w_emd  = [mean_weights[s][0] for s in SNRs_to_test]
    w_gasf = [mean_weights[s][1] for s in SNRs_to_test]
    w_bigru= [mean_weights[s][2] for s in SNRs_to_test]

    print("\nMean Attention Gate Weights per SNR (entropy-regularized):")
    print(f"{'SNR':>6}  {'a1 (EMD)':>10}  {'a2 (GASF)':>11}  {'a3 (BiGRU)':>12}")
    for snr, e, g, b in zip(SNRs_to_test, w_emd, w_gasf, w_bigru):
        print(f"{snr:>6}  {e:>10.3f}  {g:>11.3f}  {b:>12.3f}")

    # ── Adaptivity check (R4 pt.1, R3 pt.1) ───────────────────────────────
    range_emd  = max(w_emd)  - min(w_emd)
    range_gasf = max(w_gasf) - min(w_gasf)
    range_bigru= max(w_bigru)- min(w_bigru)
    max_range  = max(range_emd, range_gasf, range_bigru)
    ADAPTIVE_THRESHOLD = 0.03  # weight must vary by >3pp across SNR to claim adaptivity

    print(f"\nGate weight range across SNR: a1={range_emd:.4f}, "
          f"a2={range_gasf:.4f}, a3={range_bigru:.4f}")
    if max_range >= ADAPTIVE_THRESHOLD:
        print(f"RESULT: Gate IS SNR-dependent (max range {max_range:.4f} >= "
              f"{ADAPTIVE_THRESHOLD}). 'SNR-adaptive' language is now substantiated.")
        GATE_IS_ADAPTIVE = True
    else:
        print(f"RESULT: Gate weights remain effectively constant (max range "
              f"{max_range:.4f} < {ADAPTIVE_THRESHOLD}).")
        print("ACTION REQUIRED: Replace 'SNR-adaptive attention' language in the "
              "manuscript (abstract, Sec 3.6, Introduction) with "
              "'learned fixed cross-path weighting', per R4 pt.1 / R3 pt.1.")
        GATE_IS_ADAPTIVE = False

    # ── PLOT — Attention Weight Visualization ─────────────────────────────
    x = np.arange(len(SNRs_to_test))
    width = 0.25
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.bar(x - width, w_emd,  width, label='a₁ EMD',   color='#2E4057', alpha=0.85)
    ax1.bar(x,         w_gasf, width, label='a₂ GASF',  color='#48CAE4', alpha=0.85)
    ax1.bar(x + width, w_bigru,width, label='a₃ BiGRU', color='#E84855', alpha=0.85)
    ax1.set_xticks(x)
    ax1.set_xticklabels([f'{s} dB' for s in SNRs_to_test])
    ax1.set_ylabel('Mean Attention Weight')
    ax1.set_title('(a) Cross-Path Gate Weights vs SNR\n(entropy-regularized)', fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, linestyle='--', alpha=0.4, axis='y')
    ax1.set_ylim(0, 0.75)
    for i, (e, g, b) in enumerate(zip(w_emd, w_gasf, w_bigru)):
        ax1.text(i - width, e + 0.01, f'{e:.2f}', ha='center', fontsize=8)
        ax1.text(i,         g + 0.01, f'{g:.2f}', ha='center', fontsize=8)
        ax1.text(i + width, b + 0.01, f'{b:.2f}', ha='center', fontsize=8)

    ax2.plot(SNRs_to_test, w_emd,   marker='o', ms=8, lw=2, color='#2E4057', label='a₁ EMD')
    ax2.plot(SNRs_to_test, w_gasf,  marker='s', ms=8, lw=2, color='#48CAE4', label='a₂ GASF')
    ax2.plot(SNRs_to_test, w_bigru, marker='^', ms=8, lw=2, color='#E84855', label='a₃ BiGRU')
    ax2.set_xlabel('SNR (dB)')
    ax2.set_ylabel('Mean Attention Weight')
    ax2.set_title('(b) Attention Weight Trend across SNR', fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.grid(True, linestyle='--', alpha=0.4)
    ax2.set_xlim(-12, 22)
    ax2.set_ylim(0, 0.75)

    title_suffix = ("Entropy regularization produced SNR-dependent gating" if GATE_IS_ADAPTIVE
                     else "Weights remain near-constant — report as 'learned fixed weighting'")
    fig.suptitle(f'Fig. — Cross-Path Attention Gate Weight Distribution vs SNR\n{title_suffix}',
                 fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'p20_attention_weights_entropy_reg.png'))
    plt.show()
    print("Saved: p20_attention_weights_entropy_reg.png")
else:
    print("Skipping attention weight plot (gate model unavailable).")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# STATISTICAL SIGNIFICANCE TESTING — Proposed vs EMD-SVM (measured)
# (R1, R3 pt.2, Editor) — uses ACTUAL per-sample SVM predictions, not simulated
# ══════════════════════════════════════════════════════════════════════════
from scipy.stats import wilcoxon
from statsmodels.stats.contingency_tables import mcnemar

print("\n" + "═"*65)
print("STATISTICAL SIGNIFICANCE TESTING")
print("Wilcoxon Signed-Rank Test: Proposed vs EMD-SVM (measured)")
print("═"*65)

results_stat = []
print(f"{'SNR':>6} {'Proposed':>10} {'SVM':>8} {'W stat':>10} {'p-value':>12} {'Signif':>10}")
print("-"*65)
for snr in SNRs_to_test:
    idx = np.where(snr_test == snr)[0]
    pr  = zone_predict(models_D,
                       {"emd_input": test_d["emd_input"][idx],
                        "gaf_input": test_d["gaf_input"][idx],
                        "seq_input": test_d["seq_input"][idx]}, snr)
    proposed_correct = (np.argmax(pr, 1) == np.argmax(y_test[idx], 1)).astype(float)
    proposed_acc = proposed_correct.mean() * 100

    svm_pred = svm_clf.predict(X_te_sc[idx])
    svm_true = np.argmax(y_test[idx], axis=1)
    svm_correct = (svm_pred == svm_true).astype(float)
    svm_acc = svm_correct.mean() * 100

    diff = proposed_correct - svm_correct
    if np.all(diff == 0):
        print(f"{snr:>6}   {proposed_acc:>7.1f}%  {svm_acc:>6.1f}%  {'N/A':>10}  {'1.000':>12}  {'—':>10}")
        results_stat.append((snr, proposed_acc, svm_acc, None, 1.0, False))
        continue
    try:
        stat, p = wilcoxon(proposed_correct, svm_correct, alternative='two-sided')
        sig = p < 0.05
        print(f"{snr:>6}   {proposed_acc:>7.1f}%  {svm_acc:>6.1f}%  {stat:>10.1f}  {p:>12.4f}  {'YES ***' if p<0.001 else 'YES *' if sig else 'No':>10}")
        results_stat.append((snr, proposed_acc, svm_acc, stat, p, sig))
    except Exception as e:
        print(f"{snr:>6}   {proposed_acc:>7.1f}%  {svm_acc:>6.1f}%  {'error':>10}  {str(e)[:12]:>12}  {'?':>10}")

print("═"*65)

# ── McNemar's test at -10 dB ──────────────────────────────────────────────
print("\nMcNemar's Test at -10 dB (paired binary comparison, measured):")
snr = -10
idx = np.where(snr_test == snr)[0]
pr  = zone_predict(models_D,
                   {"emd_input": test_d["emd_input"][idx],
                    "gaf_input": test_d["gaf_input"][idx],
                    "seq_input": test_d["seq_input"][idx]}, snr)
p_correct = (np.argmax(pr, 1) == np.argmax(y_test[idx], 1))
svm_pred  = svm_clf.predict(X_te_sc[idx])
s_correct = (svm_pred == np.argmax(y_test[idx], axis=1))
b = np.sum(~p_correct &  s_correct)
c = np.sum( p_correct & ~s_correct)
table = np.array([[np.sum(~p_correct & ~s_correct), b],
                  [c, np.sum( p_correct &  s_correct)]])
try:
    res = mcnemar(table, exact=False, correction=True)
    print(f"  Contingency table: {table}")
    print(f"  Statistic: {res.statistic:.2f}, p-value: {res.pvalue:.4f}")
    print(f"  Conclusion: {'Significant (p<0.05)' if res.pvalue < 0.05 else 'Not significant'}")
except Exception as e:
    print(f"  McNemar test: {e}")

# ══════════════════════════════════════════════════════════════════════════
# PLOT: Proposed vs Real Baselines (incl. ResNet-LSTM, Transformer-AMC)
# ══════════════════════════════════════════════════════════════════════════
order = ['EMD-SVM (Baseline [1], measured)', 'kNN (k=5)', 'MLP-ANN',
         'ResNet-LSTM (reproduced)', 'Transformer-AMC (reproduced)',
         'Single-Kernel GASF+EMD', 'Dual-Kernel GASF+EMD',
         'Proposed: EMD+GASF+BiGRU']
plot_cfg = {
    'EMD-SVM (Baseline [1], measured)': ('o','--','#888888', 8),
    'kNN (k=5)':                        ('v',':', '#9B59B6', 6),
    'MLP-ANN':                          ('s',':', '#F39C12', 6),
    'ResNet-LSTM (reproduced)':         ('P','-.','#6A994E', 8),
    'Transformer-AMC (reproduced)':     ('X','-.','#BC6C25', 8),
    'Single-Kernel GASF+EMD':           ('^','-.','#48CAE4', 7),
    'Dual-Kernel GASF+EMD':             ('D','-', '#2E4057', 7),
    'Proposed: EMD+GASF+BiGRU':         ('*','-', '#E84855', 13),
}
fig, ax = plt.subplots(figsize=(11,6.5))
for label,(mk,ls,clr,ms) in plot_cfg.items():
    if label not in results: continue
    is_proposed = 'Proposed' in label
    ax.plot(SNRs_to_test, results[label], marker=mk, ms=ms,
            lw=2.8 if is_proposed else 1.8, color=clr, linestyle=ls,
            label=label, zorder=5 if is_proposed else 2)
ax.set_xlabel('Signal-to-Noise Ratio (dB)')
ax.set_ylabel('Overall Accuracy (%)')
ax.set_title('Classification Accuracy vs SNR — Proposed vs Reproduced Baselines\n'
              '(ResNet-LSTM, Transformer-AMC, EMD-SVM all measured on identical split)',
              fontweight='bold')
ax.set_xlim(-12, 22); ax.set_ylim(20, 115)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(fontsize=8.5, loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'p01_accuracy_vs_snr_real_baselines.png'))
plt.show()
print("Saved: p01_accuracy_vs_snr_real_baselines.png")

# ══════════════════════════════════════════════════════════════════════════
# PLOT: Fading Channel Robustness Bar Chart
# ══════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9,6))
fade_labels = ['Proposed\n(AWGN-trained)', 'ResNet-LSTM\n(AWGN-trained)', 'Transformer-AMC\n(AWGN-trained)']
ray_vals = [fading_results['Proposed (AWGN-trained) — Rayleigh'],
            fading_results['ResNet-LSTM (AWGN-trained) — Rayleigh'],
            fading_results['Transformer-AMC (AWGN-trained) — Rayleigh']]
ric_vals = [fading_results['Proposed (AWGN-trained) — Rician'],
            fading_results['ResNet-LSTM (AWGN-trained) — Rician'],
            fading_results['Transformer-AMC (AWGN-trained) — Rician']]
awgn_ref = [np.mean(results['Proposed: EMD+GASF+BiGRU']),
            np.mean(results['ResNet-LSTM (reproduced)']),
            np.mean(results['Transformer-AMC (reproduced)'])]
x = np.arange(len(fade_labels)); width = 0.25
ax.bar(x - width, awgn_ref, width, label='AWGN (mean over SNR)', color='#888888')
ax.bar(x,         ray_vals, width, label='Rayleigh fading', color='#48CAE4')
ax.bar(x + width, ric_vals, width, label=f'Rician fading (K={RICIAN_K_FACTOR}dB)', color='#E84855')
ax.set_xticks(x); ax.set_xticklabels(fade_labels)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Multipath Fading Channel Robustness (AWGN-trained, fading-tested)\n'
              'Editor pt.1 / R4 pt.5 / R3 pt.3', fontweight='bold')
ax.legend(); ax.grid(True, axis='y', linestyle='--', alpha=0.4)
ax.set_ylim(0, 110)
for i, (a,r,ri) in enumerate(zip(awgn_ref, ray_vals, ric_vals)):
    ax.text(i-width, a+1, f'{a:.1f}', ha='center', fontsize=9)
    ax.text(i,       r+1, f'{r:.1f}', ha='center', fontsize=9)
    ax.text(i+width, ri+1,f'{ri:.1f}',ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'p21_fading_robustness.png'))
plt.show()
print("Saved: p21_fading_robustness.png")

# ══════════════════════════════════════════════════════════════════════════
# SAVE RESULTS DICTIONARY FOR DOWNSTREAM NOTEBOOKS (NB2, NB3)
# ══════════════════════════════════════════════════════════════════════════
import pickle
nb1_export = {
    'results': results,
    'fading_results': fading_results,
    'snr_test': snr_test,
    'y_test': y_test,
    'SNRs_to_test': SNRs_to_test,
    'GATE_IS_ADAPTIVE': GATE_IS_ADAPTIVE,
    'gate_weights': mean_weights if gate_model_ok else None,
}
with open('/kaggle/working/nb1_export.pkl', 'wb') as f:
    pickle.dump(nb1_export, f)
print("\nSaved nb1_export.pkl (results, fading_results, gate weights) for NB2/NB3.")

# ── Save model weights for downstream notebooks ───────────────────────────
for i, m in enumerate(models_D):
    m.save_weights(f'/kaggle/working/model_D_seed{i}.weights.h5')
print(f"Saved {len(models_D)} model_D weight files.")

# ── Zip figures ────────────────────────────────────────────────────────────
import glob
pngs = glob.glob(os.path.join(OUTPUT_DIR, '*.png'))
if pngs:
    os.system(f'zip -r /kaggle/working/all_figures_nb1.zip {OUTPUT_DIR}/')
    print(f'\nZipped {len(pngs)} figures → /kaggle/working/all_figures_nb1.zip')
